# 分组聚合与组内变换

学习目标：按一个或多个键汇总小表，理解多级索引，并按组生成逐行指标、筛选记录和处理缺失分组。

前置知识：分类类型、聚合、行列索引、缺失值、Python 函数、均值与标准差。

运行环境：Python 3.12、pandas 3.0；示例按 pandas 3.0.6 的分组参数行为编写。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例均使用单元内构造的数据，后续单元沿用 pd。本环境已安装 PyArrow，默认 str 列采用该存储后端。金额均以元为单位。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 按门店汇总销售额

要知道各门店的销售总额，可以先用 groupby 指定分组键，再选择金额列并求和。相同门店的记录属于同一组，不要求它们在原表中相邻。

下面的 sales 按登记先后排列；store 是门店，product 是商品，amount_yuan 是每条销售记录的金额。

In [1]:
import pandas as pd

sales = pd.DataFrame(
    {"store": ["B", "A", "B", "A", "C", "C"],
     "product": ["tea", "tea", "coffee", "coffee", "tea", "tea"],
     "amount_yuan": [30, 10, 50, 20, 40, 40]},
    index=["R1", "R2", "R3", "R4", "R5", "R6"],
)
totals = sales.groupby("store", sort=False, observed=True, dropna=False)["amount_yuan"].sum()
print(sales)  # 六条记录，B 与 A 的记录交错出现；金额为 int64。
print(totals)  # B 为 80、A 为 30、C 为 80；结果为 int64 的 Series。
print(totals.index.tolist(), totals.shape)  # ['B', 'A', 'C'] (3,)。

   store product  amount_yuan
R1     B     tea           30
R2     A     tea           10
R3     B  coffee           50
R4     A  coffee           20
R5     C     tea           40
R6     C     tea           40
store
B    80
A    30
C    80
Name: amount_yuan, dtype: int64
['B', 'A', 'C'] (3,)


## 2 拆分、计算与合并

分组常用“拆分—应用—合并”（split-apply-combine）描述：按键确定各组，对每组计算，最后合并结果。“应用”指计算步骤，不要求调用 apply。

先沿图观察同色记录如何归入同一组，再将各组的计算结果合为摘要。

![pandas 官方分组图：按类别拆分记录，对组计算并组合摘要。](image/illustration/11-01-groupby-flow.svg)

引用 pandas 官方原图，保留原配色与内容；图中颜色区分类别，没有标注销售金额，也不表示必须把中间分组复制成多张表。版权与 BSD-3-Clause 许可见篇末。

下面继续使用 sales 的六条记录，追踪不相邻的两条 B 记录。sort=False 使组按首次出现的 B、A、C 排列，金额总和应分别为 80、30、80 元；这些是本例的结果，不是原图中的数值。

groupby 返回分组对象，get_group 查看一个组；agg 完成聚合。命名聚合写成“输出列名=(输入列名, 聚合名称)”。下面再同时计算均值与记录数，检查三个摘要的输出列名。

In [2]:
grouped = sales.groupby("store", sort=False, observed=True, dropna=False)
print(grouped.get_group("B"))  # B 组为 R1、R3，组内保持原始顺序。
summary = grouped.agg(
    total_yuan=("amount_yuan", "sum"),
    mean_yuan=("amount_yuan", "mean"),
    records=("amount_yuan", "size"),
)
print(summary)  # B、A、C；总额 80、30、80；均值 40、15、40；各有 2 条记录。
print(summary.dtypes)  # total_yuan、records 为 int64，mean_yuan 为 float64。
print(summary.shape)  # 预期：(3, 3)，每组一行。

   store product  amount_yuan
R1     B     tea           30
R3     B  coffee           50
       total_yuan  mean_yuan  records
store                                
B              80       40.0        2
A              30       15.0        2
C              80       40.0        2
total_yuan      int64
mean_yuan     float64
records         int64
dtype: object
(3, 3)


## 3 记录数、有效数与缺失键

### 3.1 size 与 count

GroupBy.size() 统计每组行数，包括该组数据列中的缺失；count() 按列统计非缺失值。这里 size 是分组对象的方法，需要括号，不是 DataFrame.size 的单元格数量属性。

下面单独构造缺失金额表。金额列是 float64，NaN 表示金额未知；门店键暂时都完整。

In [3]:
incomplete = pd.DataFrame(
    {"store": ["A", "A", "B"], "amount_yuan": [10.0, None, None]},
    index=["X1", "X2", "X3"],
)
missing_groups = incomplete.groupby("store", sort=False, observed=True, dropna=False)
print(missing_groups.size())  # A 有 2 行，B 有 1 行，dtype 为 int64。
print(missing_groups["amount_yuan"].count())  # 有效金额数量分别为 1、0。
print(missing_groups["amount_yuan"].sum(min_count=1))
# A 为 10.0，B 为 NaN；要求至少一个有效值，避免把未知金额当作实际零元。

store
A    2
B    1
dtype: int64
store
A    1
B    0
Name: amount_yuan, dtype: int64
store
A    10.0
B     NaN
Name: amount_yuan, dtype: float64


### 3.2 dropna 控制缺失分组键

groupby 的 dropna 默认 True，键缺失的记录不进入结果；False 把缺失键也作为一组。它控制的是“能否按键归组”，与金额统计是否跳过缺失是不同问题。

下面有一条门店未登记、但金额已知的记录。是否保留它，应由汇总口径明确决定。多键分组时，任一分组键缺失也会受到 dropna 影响。

In [4]:
unknown_store = pd.DataFrame(
    {"store": ["A", None, "A"], "amount_yuan": [10, 5, 20]},
    index=["U1", "U2", "U3"],
)
excluded = unknown_store.groupby("store", dropna=True, sort=False, observed=True).size()
included = unknown_store.groupby("store", dropna=False, sort=False, observed=True).size()
print(excluded)  # 仅 A 组的 2 条记录。
print(included)  # A 为 2，缺失键组为 1。
print(excluded.sum(), included.sum(), len(unknown_store))  # 预期：2 3 3。
assert included.sum() == len(unknown_store)

store
A    2
dtype: int64
store
A      2
NaN    1
dtype: int64
2 3 3


## 4 结果索引与组顺序

as_index=True 是默认值，聚合结果把分组键放到索引；对 DataFrame 分组使用 as_index=False，键保留为普通列。这个参数不负责给 transform 或 filter 改写输出索引。

sort=True 默认排序组键；sort=False 按组首次出现的顺序输出。二者都不改变组内记录的先后。下面继续使用原始 sales，区分组顺序与登记顺序。

In [5]:
flat_summary = sales.groupby(
    "store", as_index=False, sort=False, observed=True, dropna=False
).agg(total_yuan=("amount_yuan", "sum"))
sorted_totals = sales.groupby(
    "store", sort=True, observed=True, dropna=False
)["amount_yuan"].sum()
print(flat_summary)  # 普通 store 列依次为 B、A、C，总额为 80、30、80。
print(flat_summary.index.tolist(), flat_summary.shape)  # [0, 1, 2] (3, 2)。
print(sorted_totals.index.tolist())  # 预期：['A', 'B', 'C']。
print(sales.index.tolist())  # 原表仍按 R1 至 R6 排列。

  store  total_yuan
0     B          80
1     A          30
2     C          80
[0, 1, 2] (3, 2)
['A', 'B', 'C']
['R1', 'R2', 'R3', 'R4', 'R5', 'R6']


## 5 多键聚合与 MultiIndex

### 5.1 门店与商品共同分组

把列名列表传给 groupby，就用这些列的组合确定组。下面将 sales 按“门店、商品”汇总；C 门店的两条 tea 记录会合并成一组。

默认 as_index=True 时，多键结果使用 MultiIndex（多级索引）。它把多个键组成一条行标签，而不是把它们算进数据列。

In [6]:
by_product = sales.groupby(
    ["store", "product"], sort=False, observed=True, dropna=False
).agg(total_yuan=("amount_yuan", "sum"), records=("amount_yuan", "size"))
print(by_product)
# 组依次为 (B, tea)、(A, tea)、(B, coffee)、(A, coffee)、(C, tea)。
# 总额分别为 30、10、50、20、80，最后一组 records 为 2。
print(by_product.shape)  # 预期：(5, 2)，两个分组键位于行索引。

               total_yuan  records
store product                     
B     tea              30        1
A     tea              10        1
B     coffee           50        1
A     coffee           20        1
C     tea              80        2
(5, 2)


### 5.2 层级、层名与元组标签

MultiIndex 的每条完整标签可看成一个元组。这里第 0 层表示门店，第 1 层表示商品；层名是 store、product，具体标签则是 ("B", "tea") 等值，两者不要混淆。

nlevels 返回层数，names 返回层名。读取指定组时，loc 的行选择使用完整元组，逗号后的另一项仍是列选择。下面继续查看 by_product。

In [7]:
print(type(by_product.index).__name__)  # 预期：MultiIndex。
print(by_product.index.nlevels, list(by_product.index.names))  # 2 ['store', 'product']。
print(by_product.index.tolist())  # 五条完整元组标签，与上一单元组顺序一致。
print(by_product.loc[("C", "tea"), "total_yuan"])  # 预期：80。
print(by_product.index.is_unique)  # 预期：True，每个键组合对应一行聚合结果。

MultiIndex
2 ['store', 'product']
[('B', 'tea'), ('A', 'tea'), ('B', 'coffee'), ('A', 'coffee'), ('C', 'tea')]
80
True


### 5.3 把分组键转回列

reset_index 默认把索引层转换为普通列，再建立默认整数索引；不再保留键时才使用 drop=True。对于需要导出或继续按列处理的摘要，先转回列通常更直观。

下面把 by_product 的两个层都转为列；这个操作只改变组织方式，不重新聚合，也不增加记录。

In [8]:
product_table = by_product.reset_index()
print(product_table)  # store、product 变为前两列，总额和 records 保持原值。
print(product_table.columns.tolist(), product_table.shape)
# ['store', 'product', 'total_yuan', 'records'] (5, 4)。
print(product_table.index.tolist())  # 预期：[0, 1, 2, 3, 4]。
print(product_table["total_yuan"].tolist() == by_product["total_yuan"].tolist())  # True。

  store product  total_yuan  records
0     B     tea          30        1
1     A     tea          10        1
2     B  coffee          50        1
3     A  coffee          20        1
4     C     tea          80        2
['store', 'product', 'total_yuan', 'records'] (5, 4)
[0, 1, 2, 3, 4]
True


### 5.4 按已有索引层分组

已理解层级后，可以用 groupby(level="store") 对门店层再次分组；也可以用层位置 0。level 与 by 不要同时指定。

下面沿用仍保留 MultiIndex 的 by_product，将各商品总额再次合计为门店总额。总额可相加；若摘要存的是组均值，不能不考虑组大小就直接再求平均。

In [9]:
by_level = by_product.groupby(
    level="store", sort=False, observed=True, dropna=False
)["total_yuan"].sum()
print(by_level)  # B 为 80、A 为 30、C 为 80，dtype 为 int64。
print(by_level.index.tolist(), by_level.tolist() == totals.tolist())  # ['B', 'A', 'C'] True。
assert by_level.tolist() == totals.tolist()

store
B    80
A    30
C    80
Name: total_yuan, dtype: int64
['B', 'A', 'C'] True


## 6 未观测分类与 observed

分类列可以声明一些尚未出现在数据中的类别。pandas 3 默认 observed=True，只输出实际观测到的分类组；False 会纳入未观测类别。这个参数只在分组键含分类类型时起作用，和键本身缺失是两回事。

下面定义 east、west、north 三个区域，但没有 north 的销售记录。空组的行数是 0，均值是缺失；不要把未观测类别解释成发生过零元销售。

In [10]:
regions = pd.DataFrame({
    "region": pd.Categorical(["east", "east", "west"], categories=["east", "west", "north"]),
    "amount_yuan": [10, 20, 30],
})
observed_only = regions.groupby("region", observed=True, sort=True, dropna=False).size()
all_categories = regions.groupby("region", observed=False, sort=True, dropna=False).agg(
    records=("amount_yuan", "size"), mean_yuan=("amount_yuan", "mean")
)
print(observed_only)  # 只有 east、west，数量分别为 2、1。
print(all_categories)  # north 为 0 条记录、NaN 均值。
print(len(observed_only), len(all_categories))  # 预期：2 3。
print(all_categories.dtypes)  # records 为 int64，mean_yuan 为 float64。

region
east    2
west    1
dtype: int64
        records  mean_yuan
region                    
east          2       15.0
west          1       30.0
north         0        NaN
2 3
records        int64
mean_yuan    float64
dtype: object


## 7 transform 返回逐行结果

### 7.1 把组均值放回每条记录

agg 通常每组返回一个摘要；GroupBy.transform 返回与原数据索引对应的结果。若计算得到一个组标量，例如均值，transform 会把它重复到该组的每条记录。

下面继续使用原始 sales，计算每条记录与所在门店均值的差。这里 GroupBy.transform("mean") 可以广播组均值，不等同于未分组 DataFrame.transform 的调用条件。

In [11]:
amount_groups = sales.groupby(
    "store", sort=False, observed=True, dropna=False
)["amount_yuan"]
group_mean = amount_groups.transform("mean")
deviation = sales["amount_yuan"] - group_mean
print(group_mean.tolist())  # 预期：[40.0, 15.0, 40.0, 15.0, 40.0, 40.0]。
print(deviation)  # R1 至 R6 为 -10、-5、10、5、0、0，dtype 为 float64。
print(group_mean.index.equals(sales.index), group_mean.shape)  # True (6,)。

[40.0, 15.0, 40.0, 15.0, 40.0, 40.0]
R1   -10.0
R2    -5.0
R3    10.0
R4     5.0
R5     0.0
R6     0.0
Name: amount_yuan, dtype: float64
True (6,)


### 7.2 组内标准化与零方差

将组内离均差除以该组标准差，可以比较记录相对本组的偏离程度。本例使用 ddof=0，以该组全部记录数作分母；这里没有缺失金额。

若一组金额完全相同，标准差为 0，不能直接作为分母。本例的处理约定是把这些组的标准化结果标为 NaN，并保留原记录，供后续区分“无法标准化”和“恰好等于均值”。下面沿用 amount_groups 和 group_mean。

In [12]:
group_std = amount_groups.transform("std", ddof=0)
valid_std = group_std.where(group_std > 0)
standardized = (sales["amount_yuan"] - group_mean) / valid_std
print(group_std.tolist())  # 预期：[10.0, 5.0, 10.0, 5.0, 0.0, 0.0]。
print(standardized)  # R1、R2 为 -1，R3、R4 为 1，C 组的 R5、R6 为 NaN。
print(standardized.index.equals(sales.index), standardized.isna().sum())  # True 2。
assert standardized.loc[["R5", "R6"]].isna().all()

[10.0, 5.0, 10.0, 5.0, 0.0, 0.0]


R1   -1.0
R2   -1.0
R3    1.0
R4    1.0
R5    NaN
R6    NaN
Name: amount_yuan, dtype: float64
True 2


## 8 filter 按组保留原记录

GroupBy.filter 把每个组交给函数，函数返回一个 True 或 False，决定整组记录是否保留；它不会只返回组摘要，也不会只筛出组内的部分行。不要与按行布尔筛选或 DataFrame.filter 的标签筛选混淆。

下面保留销售总额至少 70 元的门店的全部记录。函数只读取数据，不修改传入的组；这种条件也可用 transform 与行筛选组合完成。

In [13]:
eligible = sales.groupby(
    "store", sort=False, observed=True, dropna=False
).filter(lambda group: group["amount_yuan"].sum() >= 70)
print(eligible)  # 保留 B、C 的 R1、R3、R5、R6，仍按原表顺序排列。
print(eligible.shape, eligible.index.tolist())  # (4, 3)，标签如上。
row_totals = amount_groups.transform("sum")
print(eligible.equals(sales.loc[row_totals >= 70]))  # 预期：True。
# filter 自己也有 dropna 参数；默认丢弃不通过条件的组，与 groupby 的缺失键参数不同。

   store product  amount_yuan
R1     B     tea           30
R3     B  coffee           50
R5     C     tea           40
R6     C     tea           40


(4, 3) ['R1', 'R3', 'R5', 'R6']
True


## 9 选学：apply 与分组键

### 9.1 返回每组不同数量的行

已有聚合、transform 或其他内置分组操作能完成任务时，优先使用它们。apply 的函数接收一组 DataFrame，可返回标量、Series 或 DataFrame，再由 pandas 合并，适合需要自定义输出结构的操作。

下面的任务按登记顺序，从每个门店取“累计金额首次达到该店总额一半”的最短前缀。金额均为正数；达到阈值之后的记录不再需要，因此各组返回的行数可以不同。pandas 3 中 include_groups=False，传入函数的表不包含用作分组键的 store 列。

In [14]:
def first_half(group):
    # include_groups=False 排除了分组列，函数直接计算金额。
    target = group["amount_yuan"].sum() / 2
    before_current = group["amount_yuan"].cumsum() - group["amount_yuan"]
    return group.loc[before_current < target, ["product", "amount_yuan"]]


prefixes = sales.groupby(
    "store", group_keys=True, sort=False, observed=True, dropna=False
).apply(first_half, include_groups=False)
print(prefixes)
# B 组取 R1、R3，A 组取 R2、R4，C 组只取 R5；共五行两列。
print(prefixes.index.nlevels, prefixes.shape)  # 2 (5, 2)：门店键与原行标签组成 MultiIndex。

         product  amount_yuan
store                        
B     R1     tea           30
      R3  coffee           50
A     R2     tea           10
      R4  coffee           20
C     R5     tea           40
2 (5, 2)


### 9.2 group_keys 与 include_groups

group_keys 控制 apply 合并上述逐行结果时是否把组键加进结果索引；include_groups 控制传入函数的数据是否包含分组列，二者职责不同。pandas 3 不再允许 include_groups=True。

下面仍使用 first_half；去掉结果中的组键不会自动把所有行重新排列成原表顺序，也不会把 store 补回普通列。

In [15]:
plain_prefixes = sales.groupby(
    "store", group_keys=False, sort=False, observed=True, dropna=False
).apply(first_half, include_groups=False)
print(plain_prefixes.index.tolist())  # 预期：['R1', 'R3', 'R2', 'R4', 'R5']。
print(plain_prefixes.columns.tolist())  # 预期：['product', 'amount_yuan']。

['R1', 'R3', 'R2', 'R4', 'R5']
['product', 'amount_yuan']


In [16]:
# 预期 ValueError：pandas 3 已不允许 include_groups=True。
sales.groupby("store", observed=True).apply(first_half, include_groups=True)

ValueError: include_groups=True is no longer allowed.

## 10 选学：组内排名与顺序运算

### 10.1 排名与累计

分组 rank 只在各组内部比较，返回与原记录对应的名次；并列规则应显式选择。分组 cumsum 只在同组内累计，组内顺序仍来自输入。

下面沿用 sales 的登记先后顺序。groupby 的 sort 参数只控制组键顺序，不能代替按业务时间预先排列记录。

In [17]:
print(amount_groups.rank(ascending=False, method="min"))
# R1、R2 名次为 2；R3、R4 为 1；C 组两条均为 1，dtype 为 float64。
print(amount_groups.cumsum())
# R1 至 R6 为 30、10、80、30、40、80，dtype 为 int64。
print(amount_groups.cumsum().index.equals(sales.index))  # 预期：True。

R1    2.0
R2    2.0
R3    1.0
R4    1.0
R5    1.0
R6    1.0
Name: amount_yuan, dtype: float64
R1    30
R2    10
R3    80
R4    30
R5    40
R6    80
Name: amount_yuan, dtype: int64
True


### 10.2 上一条记录与差值

分组 shift(1) 取同组前一条记录的值；diff() 计算本条与同组前一条的差。每组的第一条没有前项，结果为缺失，不能跨组拿其他门店的记录补齐。

下面仍使用 sales 的登记顺序；“上一条”是同一门店在输入中的上一条，不一定是上一天。

In [18]:
previous = amount_groups.shift(1)
change = amount_groups.diff()
print(previous)  # R1、R2、R5 为 NaN；R3、R4、R6 分别为 30、10、40。
print(change)  # 同三个首项为 NaN；R3、R4、R6 分别为 20、10、0。
print(previous.dtype, change.dtype)  # 预期：float64 float64，整数输入因缺失转为浮点结果。

R1     NaN
R2     NaN
R3    30.0
R4    10.0
R5     NaN
R6    40.0
Name: amount_yuan, dtype: float64
R1     NaN
R2     NaN
R3    20.0
R4    10.0
R5     NaN
R6     0.0
Name: amount_yuan, dtype: float64
float64 float64


## 11 选学：用 Grouper 按月份分组

Grouper 用来描述更具体的分组规则。key 指定日期列，freq="MS" 按月初划分；本例指定 closed="left"、label="left"，表示包含左端、不包含下月月初，结果用本月月初标记。

下面是无时区的本地日期，先用 to_datetime 按明确的“年-月-日”格式解析。2 月 1 日恰好属于二月组，不计入一月。

In [19]:
dated = pd.DataFrame({
    "day": pd.to_datetime(["2026-01-01", "2026-01-31", "2026-02-01"], format="%Y-%m-%d"),
    "amount_yuan": [10, 20, 30],
})
monthly = dated.groupby(
    pd.Grouper(key="day", freq="MS", closed="left", label="left"),
    sort=True, observed=True, dropna=False,
)["amount_yuan"].sum()
print(monthly)  # 2026-01-01 对应 30，2026-02-01 对应 30；金额 dtype 为 int64。
print(monthly.shape)  # 预期：(2,)，索引为月份边界的 DatetimeIndex。

day
2026-01-01    30
2026-02-01    30
Freq: MS, Name: amount_yuan, dtype: int64
(2,)


## 本章小结

（1）分组先确定键和有效记录口径；size 统计行数，count 统计各列有效值，dropna 控制缺失键。

（2）agg 生成组摘要；transform 把组内计算对应回原记录；filter 决定是否保留整组记录。

（3）多键结果通常使用 MultiIndex，完整标签是键值元组；先认识层和层名，再用 reset_index 或按 level 分组。

（4）as_index、sort、observed 分别影响键的位置、组顺序和未观测类别，不能替代彼此。

（5）apply 的输出索引与函数输入分别由 group_keys、include_groups 控制；组内累计和前后差值依赖明确的输入顺序。

## 练习

（1）先预测下面两个结果的组数、行数统计与有效值统计，再运行核对。解释缺失分组键与缺失数值分别在哪一步产生影响。

In [20]:
example = pd.DataFrame({"team": ["A", "A", None], "value": [1.0, None, 3.0]})
print(example.groupby("team", dropna=True, observed=True).size())
print(example.groupby("team", dropna=False, observed=True)["value"].count())
# 运行前写下预测；运行后分别核对被纳入分组的行数和每组有效数值个数。

team
A    2
dtype: int64
team
A      1
NaN    1
Name: value, dtype: int64


（2）按“地区、商品”生成金额总和，查看 MultiIndex 层数、层名与完整元组标签。读取 ("east", "tea") 的金额，再用 reset_index 转成普通表，最后从保留 MultiIndex 的结果按地区层求和。

In [21]:
orders = pd.DataFrame({"region": ["east", "east", "west", "east"],
                       "product": ["tea", "coffee", "tea", "tea"],
                       "amount_yuan": [10, 20, 30, 40]})
# 在此完成命名聚合及索引检查，显式指定 sort、observed、dropna。
# 检查：三组总额分别为 east/tea 50、east/coffee 20、west/tea 30。
# reset_index 后两个键成为列；再按 region 层求和得到 east 70、west 30。

（3）任务原本只要每队平均分，随后改成给每条原记录附上本队平均分。应分别选择 agg 还是 transform？写出两种结果并解释形状与索引差别。

再加一个条件：保留平均分至少 80 的队伍的所有原记录，说明 filter 的返回值为什么又不同。

In [22]:
scores = pd.DataFrame({"team": ["A", "B", "A", "B"], "score": [70, 80, 90, 60]},
                      index=["P1", "P2", "P3", "P4"])
# 在此完成摘要、逐行均值与整组筛选，并说明方法选择理由。
# 检查：A、B 平均分为 80、70；逐行均值依次 80、70、80、70，保留 P1 至 P4。
# 最后一项保留 A 队的 P1、P3，不能只留下分数本身大于等于 80 的记录。

（4）分类列声明三个班级，但只有两个班级出现。分别输出仅观测班级和全部声明班级的行数；再为每行生成组内标准化值，规定标准差为 0 的班级结果为缺失，并解释该处理规则。

In [23]:
classes = pd.DataFrame({
    "class": pd.Categorical(["A", "A", "B", "B"], categories=["A", "B", "C"]),
    "score": [60, 80, 90, 90],
})
# 在此分别设置 observed，并用 transform 生成均值和 ddof=0 的标准差。
# 检查：observed=True 的行数结果为 A 2、B 2；False 额外有 C 0。
# 标准化后 A 两行为 -1、1，B 两行为 NaN；输出与原表保持四行及相同索引。

### 重点练习提示（第 3 题）

提示一：先写出每个任务期待的行数：每组一行、每条输入一行，还是保留整组原记录。

提示二：分别用 agg、transform 与 filter；整组筛选的判断是组均值而不是单条成绩。

### 参考解析（第 3 题）

按 team 分组并显式指定 sort、observed、dropna 后，score.agg("mean") 得到 A=80、B=70，是两组摘要；score.transform("mean") 得到 P1 至 P4 对应的 80、70、80、70，保留原四行索引。filter 的函数判断组内 score.mean() 是否至少为 80，保留 P1、P3 两条 A 队原记录；P1 自己只有 70 分，也必须保留。若直接筛 score>=80，会错误保留 B 队 P2 并丢掉 P1。

## 参考与引用来源

本章引用 pandas 官方分组原图，版权和 BSD-3-Clause 许可见下表；该图用于说明分组关系，不是运行截图。

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方在线文档（课程基线 3.0.6） | [Group by: split-apply-combine](https://pandas.pydata.org/docs/user_guide/groupby.html) 的 Splitting an object into groups、GroupBy sorting、GroupBy dropna、Named aggregation、Transformation、Filtration、Flexible apply：分组流程与接口选择；[DataFrame.groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)、[api.typing.DataFrameGroupBy.agg](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.agg.html)、[api.typing.DataFrameGroupBy.size](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.size.html)、[api.typing.DataFrameGroupBy.count](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.count.html)、[api.typing.DataFrameGroupBy.sum](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.sum.html) 的键、as_index、sort、observed、dropna、返回结构、min_count；[MultiIndex / advanced indexing](https://pandas.pydata.org/docs/user_guide/advanced.html) 的 Creating a MultiIndex、Advanced indexing：元组标签及读取；[MultiIndex.nlevels](https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.nlevels.html)、[MultiIndex.names](https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.names.html)、[DataFrame.reset_index](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html) 的层级与键转列；[api.typing.SeriesGroupBy.transform](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.SeriesGroupBy.transform.html)、[api.typing.SeriesGroupBy.std](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.SeriesGroupBy.std.html)、[Series.where](https://pandas.pydata.org/docs/reference/api/pandas.Series.where.html)、[api.typing.DataFrameGroupBy.filter](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.filter.html) 的同索引结果、标量广播、ddof、条件保留与整组筛选；[api.typing.DataFrameGroupBy.apply](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.apply.html) 的 include_groups 在 3.0 的变化、group_keys 示例及不得修改组对象；[api.typing.SeriesGroupBy.rank](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.SeriesGroupBy.rank.html)、[api.typing.SeriesGroupBy.cumsum](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.SeriesGroupBy.cumsum.html)、[api.typing.SeriesGroupBy.shift](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.SeriesGroupBy.shift.html)、[api.typing.SeriesGroupBy.diff](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.SeriesGroupBy.diff.html) 的组内排名、累计与顺序运算；[Grouper](https://pandas.pydata.org/docs/reference/api/pandas.Grouper.html)、[to_datetime](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) 的 key、freq、closed、label、format；[Time series — Offset aliases](https://pandas.pydata.org/docs/user_guide/timeseries.html#offset-aliases) 的 MS；[字符串迁移指南](https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#background) 的默认 str 与 PyArrow 存储。  图源（官方文档 3.0.6，2026-09-22 核查）：[How to calculate summary statistics](https://pandas.pydata.org/docs/getting_started/intro_tutorials/06_calculate_statistics.html#aggregating-statistics-grouped-by-category) 的 Aggregating statistics grouped by category；原图 [06_groupby.svg](https://pandas.pydata.org/docs/_images/06_groupby.svg)。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[groupby](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/groupby.rst)、[advanced](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/advanced.rst)、[timeseries](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/timeseries.rst)、[migration-3-strings](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/migration-3-strings.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。  图源版权与许可：[pandas v3.0.6 LICENSE](https://github.com/pandas-dev/pandas/blob/v3.0.6/LICENSE)，完整 BSD-3-Clause 条款。 |